# Machine Learning - Practical 09: Debugging and Error Analysis

**Course:** Machine Learning - National University of Kyiv-Mohyla Academy (NaUKMA)

**Instructor:** Dmytro Kuzmenko - kuzmenko@ukma.edu.ua

| | |
|---|---|
| Week | 9 |
| Module | 9. Diagnosing Machine-Learning Systems |
| Format | Practical session (not graded - exam preparation) |
| Estimated time | 1.5-2 h of active work including discussion |
| Prerequisites | P01-P08; homework HA6 recommended |


## AI Use Disclosure

Fill this in before submitting (see course policy).

| Field | Your entry |
|---|---|
| AI tools used | |
| Nature of assistance | |
| Representative prompts or relevant interaction | |
| What I independently verified or changed | |

## Learning Objectives

- Diagnose bias vs variance from learning curves (training-set-size sweep).
- Perform error analysis on misclassified examples: confusion matrix, per-class metrics, and example inspection.
- Compute and interpret permutation importance with its caveats.
- Detect distribution shift with a train-region vs shifted-region experiment, and diagnose "what happened" cases from train/validation evidence.

## Warm-up (10 min)

### Question 1 (multiple choice)

A model reports train accuracy 0.99 and validation accuracy 0.62. The most likely diagnosis is:

- A. The model underfits: it is too simple for the data.
- B. The model overfits: it memorizes the training set and does not generalize.
- C. The validation set is too large.
- D. Nothing is wrong; this gap is normal.


**Your answer:**

### Question 2 (quick reasoning)

You inherit a trained model and a single number: validation accuracy 0.83. Which three additional numbers would you ask for before judging the model?


**Your answer:**

### Question 3 (mini-interpretation)

A digit classifier has overall accuracy 0.97, but recall for digit 1 is 0.35 and most of its errors are predicted as 8. What does this tell you about where the model fails, and what would you inspect next?


**Your answer:**

## Guided Exercise (60 min)

We diagnose models with learning curves, error analysis, permutation importance, and a distribution-shift experiment. All experiments are deterministic.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, r2_score
%matplotlib inline
np.random.seed(42)

### Task 1: Learning curves - bias vs variance

**Experiment.** A non-linear synthetic classification problem (two classes, 20 features, only 4 informative, each class made of two clusters). For a linear model (LogisticRegression) and a random forest, sweep the training-set size and record train and 5-fold CV accuracy.

In [2]:
X, y = make_classification(n_samples=900, n_features=20, n_informative=4, n_redundant=4,
                           n_clusters_per_class=2, flip_y=0.05, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
}
train_sizes = [0.1, 0.25, 0.5, 0.75, 1.0]
curves = {}
for name, model in models.items():
    sizes, train_scores, val_scores = learning_curve(
        model, Xtr, ytr, cv=5, train_sizes=train_sizes, random_state=42)
    curves[name] = (train_scores.mean(axis=1), val_scores.mean(axis=1))
    print(name)
    print("  train:", np.round(train_scores.mean(axis=1), 3))
    print("  val:  ", np.round(val_scores.mean(axis=1), 3))

LogisticRegression
  train: [0.844 0.787 0.751 0.741 0.742]
  val:   [0.665 0.686 0.701 0.702 0.714]


RandomForest
  train: [1. 1. 1. 1. 1.]
  val:   [0.613 0.723 0.719 0.75  0.761]


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (name, (tr, va)) in zip(axes, curves.items()):
    ax.plot(sizes, tr, marker="o", label="train")
    ax.plot(sizes, va, marker="o", label="validation (5-fold CV)")
    ax.set_title(name)
    ax.set_xlabel("training-set fraction")
    ax.set_ylabel("accuracy")
    ax.set_ylim(0.5, 1.0)
    ax.legend()
plt.tight_layout()

**Diagnosis.** Which model shows the signature of high variance (large train/validation gap that closes as data grows) and which shows high bias (both curves low and converging)?


**Your answer:**

**Evidence.** Use the actual numbers: at which training-set size is the gap largest for the random forest, and at which size do the logistic curves meet?


**Your answer:**

**Counterfactual.** Predict how the random-forest curves change if the dataset is doubled to 1800 samples, and how the logistic curves change if the true boundary were linear.


**Your answer:**

### Task 2: Error analysis on the digits dataset

**Experiment.** Train a deliberately weak model - a decision tree of depth 6 - on digits (64 pixel features, 10 classes). Examine the confusion matrix, per-class precision/recall, and a sample of misclassified images.

In [4]:
digits = load_digits()
X_d, y_d = digits.data, digits.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_d, y_d, test_size=0.3, stratify=y_d, random_state=42)

tree = DecisionTreeClassifier(max_depth=6, random_state=42).fit(Xd_tr, yd_tr)
pred = tree.predict(Xd_te)
print("overall accuracy: {:.3f}".format((pred == yd_te).mean()))

cm = confusion_matrix(yd_te, pred)
disp = ConfusionMatrixDisplay(cm, display_labels=digits.target_names)
disp.plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix: tree depth 6 on digits")
plt.tight_layout()

overall accuracy: 0.780


In [5]:
report = classification_report(yd_te, pred, zero_division=0, output_dict=True)
rows = []
for d in digits.target_names:
    rows.append({"digit": d,
                 "precision": round(report[str(d)]["precision"], 3),
                 "recall": round(report[str(d)]["recall"], 3),
                 "f1": round(report[str(d)]["f1-score"], 3),
                 "support": report[str(d)]["support"]})
print(pd.DataFrame(rows).to_string(index=False))

 digit  precision  recall    f1  support
     0      0.942   0.907 0.925     54.0
     1      0.864   0.345 0.494     55.0
     2      0.734   0.887 0.803     53.0
     3      0.682   0.818 0.744     55.0
     4      0.839   0.870 0.855     54.0
     5      0.980   0.873 0.923     55.0
     6      0.944   0.944 0.944     54.0
     7      0.772   0.815 0.793     54.0
     8      0.484   0.577 0.526     52.0
     9      0.707   0.759 0.732     54.0


In [6]:
mis = np.where(pred != yd_te)[0]
print("misclassified examples:", len(mis))
shown = mis[:15]
fig, axes = plt.subplots(3, 5, figsize=(9, 6))
for ax, idx in zip(axes.ravel(), shown):
    ax.imshow(Xd_te[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title("true={} pred={}".format(yd_te[idx], pred[idx]), fontsize=9)
    ax.axis("off")
fig.suptitle("Misclassified digit images")
plt.tight_layout()

misclassified examples: 119


**Interpretation.** Which class has the lowest recall, and which class absorbs most of those errors? Look at the misclassified images: is the confusion explainable by visual similarity, or does the model seem to ignore a whole style of writing?


**Your answer:**

**Decision.** Given the confusion pattern, which single change to the modeling pipeline would you test first (capacity, preprocessing, data, evaluation)? Justify with the evidence above.


**Your answer:**

### Task 3: Permutation importance

**Experiment.** Fit a random forest on the synthetic classification problem and compute permutation importance on the held-out test set: shuffle each feature and measure the drop in accuracy.

In [7]:
rf = RandomForestClassifier(n_estimators=150, random_state=42).fit(Xtr, ytr)
pi = permutation_importance(rf, Xte, yte, n_repeats=5, random_state=42, scoring="accuracy")

imp = pd.DataFrame({
    "feature": ["f{:02d}".format(i) for i in range(X.shape[1])],
    "importance": pi.importances_mean,
    "std": pi.importances_std,
}).sort_values("importance", ascending=False)
print(imp.head(8).to_string(index=False))

feature  importance      std
    f12    0.073778 0.015293
    f04    0.072889 0.013363
    f18    0.042667 0.024921
    f19    0.024889 0.009152
    f06    0.024889 0.009152
    f10    0.018667 0.005183
    f13    0.016000 0.008243
    f00    0.009778 0.003326


In [8]:
fig, ax = plt.subplots(figsize=(7, 4.5))
top = imp.head(10)
ax.barh(top["feature"][::-1], top["importance"][::-1], xerr=top["std"][::-1], capsize=3)
ax.set_xlabel("mean accuracy drop when the feature is shuffled")
ax.set_title("Permutation importance (RandomForest, 20 features)")
plt.tight_layout()

**Interpretation.** Only four features are truly informative. Do the top importances match that? What does a non-zero importance for a noise feature mean here (correlation, chance)?


**Your answer:**

**Counterfactual.** Permutation importance is computed on the test set. Predict what happens to the ranking if the test set is replaced by a shifted one (different feature distribution), and why importances are deployment-specific.


**Your answer:**

### Task 4: Distribution shift - train region vs shifted region

**Experiment.** Regression on y = sin(2*pi*t) + noise with polynomial features. Compare three numbers: random 5-fold CV R2, in-distribution R2 (same region as training), and R2 on the future region (t >= 0.8) that the model never saw.

In [9]:
rng = np.random.RandomState(42)
N = 800
t = np.linspace(0, 1, N)
y_reg = np.sin(2 * np.pi * t) + rng.normal(0, 0.05, N)
mask = t < 0.8

model = make_pipeline(PolynomialFeatures(degree=8), Ridge(alpha=1.0))
cv_r2 = cross_val_score(model, t.reshape(-1, 1), y_reg, cv=KFold(5, shuffle=True, random_state=42),
                        scoring="r2").mean()
model.fit(t[mask].reshape(-1, 1), y_reg[mask])
r2_in = r2_score(y_reg[mask], model.predict(t[mask].reshape(-1, 1)))
r2_future = r2_score(y_reg[~mask], model.predict(t[~mask].reshape(-1, 1)))
print("random 5-fold CV R2 : {:.3f}".format(cv_r2))
print("in-distribution R2  : {:.3f}".format(r2_in))
print("future region R2    : {:.3f}".format(r2_future))

random 5-fold CV R2 : 0.862
in-distribution R2  : 0.859
future region R2    : -24.541


In [10]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t, y_reg, lw=1, label="true y")
ax.plot(t, model.predict(t.reshape(-1, 1)), lw=1.5, label="polynomial fit")
ax.axvspan(0.8, 1.0, color="red", alpha=0.15, label="future region (test)")
ax.set_xlabel("t"); ax.set_ylabel("y")
ax.set_title("Polynomial fit: interpolation works, extrapolation fails")
ax.legend()
plt.tight_layout()

**Interpretation.** The random CV looks fine (~0.85), yet the model is useless on the future region. What exactly did the random folds hide, and why is this a distribution-shift problem rather than a variance problem?


**Your answer:**

**Counterfactual.** Predict what happens to the future-region R2 if the test region were t in [0.75, 0.8] instead of [0.8, 1.0], and if the true function were a straight line.


**Your answer:**

### "What happened?" case studies

For each case below, look at the train/validation numbers and the curve, then write the diagnosis and the single most useful next step. The curves are generated from the tables.

In [11]:
cases = {
    "Case A": {"epoch": np.arange(1, 31),
                "train": 0.98 - 0.35 * np.exp(-np.arange(30) / 6.0),
                "val": 0.62 + 0.05 * np.exp(-((np.arange(30) - 10) ** 2) / 18.0) - 0.003 * np.maximum(np.arange(30) - 10, 0)},
    "Case B": {"epoch": np.arange(1, 31),
                "train": 0.70 + 0.001 * np.arange(30),
                "val": 0.68 + 0.001 * np.arange(30)},
    "Case C": {"epoch": np.arange(1, 31),
                "train": np.linspace(0.72, 0.91, 30),
                "val": np.linspace(0.70, 0.89, 30)},
}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (name, d) in zip(axes, cases.items()):
    ax.plot(d["epoch"], d["train"], label="train")
    ax.plot(d["epoch"], d["val"], label="validation")
    ax.set_title(name)
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy")
    ax.set_ylim(0.45, 1.0)
    ax.legend()
plt.tight_layout()
for name, d in cases.items():
    print(name, "final train {:.2f}  final val {:.2f}".format(d["train"][-1], d["val"][-1]))

Case A final train 0.98  final val 0.56
Case B final train 0.73  final val 0.71
Case C final train 0.91  final val 0.89


**Case A.** Train accuracy keeps rising while validation accuracy peaks around epoch 10 and then drops. What is happening, and what is the most useful next step?


**Your answer:**

**Case B.** Both curves are low and flat (around 0.70), with a small gap. What is the diagnosis, and what would you change first?


**Your answer:**

**Case C.** Train and validation rise together to ~0.90. A colleague claims the model is ready for deployment. What evidence is still missing before that claim is justified?


**Your answer:**

## Discussion Questions (15 min)

1. Learning curves need both the train and the validation curve. What does each curve alone hide?
2. In Task 2 the tree has recall 0.35 for digit 1. How would you phrase this failure for a non-technical stakeholder ("the model is bad" vs a precise statement)?
3. Permutation importance vs impurity-based importance: why can the latter be misleading with correlated features, and what does permutation importance change?
4. Distribution shift: which of the two shifts (covariate shift, label shift) does the polynomial demo illustrate? Give a real-world example of each.
5. Error analysis is done on a fixed test set. Why must the test set be used only for reporting, and what happens if you iterate on the test set while fixing errors?
6. A model is 0.90 on validation but 0.55 on the deployment probe. What are the three most likely explanations, and how do you test each?

## Challenge (25 min)

### Task 5: Fix a confused model and verify the fix

The depth-6 tree from Task 2 confuses digit 1 with digit 8 (and others). Propose **one concrete fix** (e.g., more capacity, an ensemble, preprocessing, or a different model family), implement it, and compare the confusion pattern before and after. Report overall accuracy and the recall of the previously worst class.

In [12]:
# Your code

**Interpretation.** Did the fix reduce the specific confusion you targeted, or did it improve everything uniformly? What does the answer tell you about the cause of the original errors?


**Your answer:**

**Counterfactual.** If your fix improved validation accuracy but you had tuned it on the validation set repeatedly, how confident would you be in the improvement? What protocol would make the estimate honest?


**Your answer:**

## Takeaways

- Learning curves separate bias from variance: converging low curves mean high bias; a large train/validation gap that closes with data means high variance.
- Error analysis starts with the confusion matrix and per-class metrics, then moves to individual examples - the pattern of errors, not the count, drives the fix.
- Permutation importance measures the drop in performance when a feature is shuffled on the test set; it is deployment-specific and more robust than impurity-based importance.
- Random CV hides distribution shift: interpolation scores can look excellent while extrapolation fails catastrophically.
- A single accuracy number is not a diagnosis; you need train vs validation vs baseline, plus the error structure.
- Every fix should be verified against the specific failure it targets, and the test set must stay untouched until the final report.